# M5v3 — 정제 + 재학습 (자동, 핸드폰 가능)

**작업 흐름 (모든 셀을 차례로 Run):**
1. 환경 설정
2. Drive 마운트
3. 데이터 + best.pt 로드
4. **정제 (Active Learning)** ~1h
5. **Hard Sample Mining** ~30분
6. **재학습 (multi-stage)** ~6h
7. ONNX export + Drive 저장

**총 시간**: ~7-8시간 (코랩 12h 한도 안)

**Drive 준비물:**
1. `frames_ade.zip` — M5v2 v2 데이터셋 (이미 있음)
2. `m5v2_v2_autosave/best.pt` — M5v2 v2 학습 결과 (autosave에 저장됨)

**런타임:** GPU T4

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv
!pip install -q ultralytics onnx onnxslim onnxruntime-gpu numpy pillow

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

In [ ]:
import os, shutil, zipfile, json
from pathlib import Path

# ★ 본인 계정에 맞게 수정 ★
DRIVE_DIR = Path('/content/drive/MyDrive/drone_inspect')  # A 계정 폴더 소유자, C는 'drone_inspect_A'
ZIP_NAME = 'frames_ade.zip'
BEST_PT_PATH = DRIVE_DIR / 'm5v2_v2_autosave' / 'best.pt'  # 또는 m5v2_v2_results/m5v2_v2_best.pt

WORK = Path('/content/m5v3')
WORK.mkdir(parents=True, exist_ok=True)

# 1) 데이터셋 압축 해제
zip_path = DRIVE_DIR / ZIP_NAME
assert zip_path.exists(), f'{zip_path} 없음'
ds_dir = WORK / 'frames_ade'
if not (ds_dir / 'images' / 'val').exists():
    if ds_dir.exists():
        shutil.rmtree(ds_dir, ignore_errors=True)
    print('Extracting...')
    with zipfile.ZipFile(zip_path) as z:
        for m in z.namelist():
            rel = m.replace('\\', '/')
            target = WORK / rel
            if rel.endswith('/'):
                target.mkdir(parents=True, exist_ok=True)
            else:
                target.parent.mkdir(parents=True, exist_ok=True)
                with z.open(m) as src, open(target, 'wb') as dst:
                    shutil.copyfileobj(src, dst)

# 2) best.pt 확인
if not BEST_PT_PATH.exists():
    # autosave 안 됐을 경우 results 폴더 시도
    alt = DRIVE_DIR / 'm5v2_v2_results' / 'm5v2_v2_best.pt'
    if alt.exists():
        BEST_PT_PATH = alt
    else:
        raise FileNotFoundError(f'M5v2 v2 best.pt 못 찾음: {BEST_PT_PATH} or {alt}')
best_pt = WORK / 'baseline_best.pt'
shutil.copy2(BEST_PT_PATH, best_pt)
print(f'baseline best.pt: {best_pt} ({best_pt.stat().st_size/1024/1024:.1f}MB)')
for s in ['train','val','test']:
    p = ds_dir / 'images' / s
    if p.exists():
        print(f'  {s}: {len(list(p.glob("*")))} images')

## 4. 데이터 정제 (Active Learning)

잘못된 GT 제거 + 놓친 GT 추가.

In [ ]:
import numpy as np
from ultralytics import YOLO

IOU_BAD = 0.3
CONF_HIGH = 0.85
MAX_MISSED_PER_IMG = 5

def yolo_to_xyxy(box, w, h):
    cx, cy, bw, bh = box[1]*w, box[2]*h, box[3]*w, box[4]*h
    return [cx-bw/2, cy-bh/2, cx+bw/2, cy+bh/2]

def iou(a, b):
    x1, y1 = max(a[0], b[0]), max(a[1], b[1])
    x2, y2 = min(a[2], b[2]), min(a[3], b[3])
    inter = max(0, x2-x1) * max(0, y2-y1)
    if inter == 0: return 0.0
    aa = (a[2]-a[0])*(a[3]-a[1])
    bb = (b[2]-b[0])*(b[3]-b[1])
    return inter / (aa + bb - inter)

def read_labels(p):
    if not p.exists(): return []
    boxes = []
    for line in p.read_text().splitlines():
        parts = line.strip().split()
        if len(parts) >= 5:
            try: boxes.append([int(parts[0])] + [float(x) for x in parts[1:5]])
            except: pass
    return boxes

def write_labels(p, boxes):
    if not boxes:
        if p.exists(): p.unlink()
        return
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text('\n'.join(f'{b[0]} {b[1]:.6f} {b[2]:.6f} {b[3]:.6f} {b[4]:.6f}' for b in boxes))

refined = WORK / 'frames_refined'
if refined.exists(): shutil.rmtree(refined)
refined.mkdir()

model = YOLO(str(best_pt))
stats = {'total':0, 'bad':0, 'missed':0}

for split in ['train', 'val', 'test']:
    src_img = ds_dir / 'images' / split
    src_lbl = ds_dir / 'labels' / split
    if not src_img.exists(): continue
    dst_img = refined / 'images' / split
    dst_lbl = refined / 'labels' / split
    dst_img.mkdir(parents=True, exist_ok=True)
    dst_lbl.mkdir(parents=True, exist_ok=True)
    
    images = [f for f in src_img.iterdir() if f.suffix.lower() in {'.jpg','.jpeg','.png'}]
    print(f'[{split}] {len(images)} images')
    
    if split in ['val', 'test']:
        # val/test는 그대로 복사
        for im in images:
            shutil.copy2(im, dst_img / im.name)
            l = src_lbl / (im.stem + '.txt')
            if l.exists(): shutil.copy2(l, dst_lbl / l.name)
        continue
    
    BATCH = 16
    for i in range(0, len(images), BATCH):
        batch = images[i:i+BATCH]
        results = model.predict(source=[str(b) for b in batch], conf=0.25, iou=0.5, imgsz=1280, verbose=False, save=False)
        for img_path, res in zip(batch, results):
            stats['total'] += 1
            gt = read_labels(src_lbl / (img_path.stem + '.txt'))
            pred_xyxy = res.boxes.xyxy.cpu().numpy() if len(res.boxes)>0 else np.empty((0,4))
            pred_conf = res.boxes.conf.cpu().numpy() if len(res.boxes)>0 else np.empty(0)
            pred_cls = res.boxes.cls.cpu().numpy().astype(int) if len(res.boxes)>0 else np.empty(0,dtype=int)
            W, H = res.orig_shape[1], res.orig_shape[0]
            
            cleaned = []
            for g in gt:
                g_xy = yolo_to_xyxy(g, W, H)
                same = pred_cls == g[0]
                if same.sum() == 0:
                    cleaned.append(g)
                    continue
                ious = np.array([iou(g_xy, pred_xyxy[j]) for j in np.where(same)[0]])
                if ious.max() >= IOU_BAD:
                    cleaned.append(g)
                else:
                    stats['bad'] += 1
            
            missed = 0
            for j in range(len(pred_xyxy)):
                if pred_conf[j] < CONF_HIGH: continue
                matched = False
                for g in cleaned:
                    if int(g[0]) != pred_cls[j]: continue
                    if iou(yolo_to_xyxy(g, W, H), pred_xyxy[j]) >= IOU_BAD:
                        matched = True
                        break
                if not matched and missed < MAX_MISSED_PER_IMG:
                    x1,y1,x2,y2 = pred_xyxy[j]
                    cleaned.append([int(pred_cls[j]), (x1+x2)/2/W, (y1+y2)/2/H, (x2-x1)/W, (y2-y1)/H])
                    stats['missed'] += 1
                    missed += 1
            
            shutil.copy2(img_path, dst_img / img_path.name)
            write_labels(dst_lbl / (img_path.stem + '.txt'), cleaned)
        if (i+BATCH) % 320 == 0 or i+BATCH >= len(images):
            print(f'  {min(i+BATCH, len(images))}/{len(images)} (bad:{stats["bad"]}, missed:{stats["missed"]})')

# data.yaml
yaml_text = f"""path: {refined}
train: images/train
val: images/val
test: images/test
nc: 4
names:
  0: wall_edge
  1: ceiling_edge
  2: door_frame
  3: window_frame
"""
data_yaml = WORK / 'frames_refined.yaml'
data_yaml.write_text(yaml_text)
print(f'\n=== 정제 완료 ===\n  total: {stats["total"]}, bad: {stats["bad"]}, missed: {stats["missed"]}')

## 5. Hard Sample Mining

loss 높은 샘플 추출 → 가중치 5배.

In [ ]:
# Hard sample은 별도 가중치 적용 어렵지만, ultralytics는 dataset 복사로 우회 가능
# 단순화: loss 높은 train 이미지를 5배 복사하여 데이터셋에 추가
import torch
from PIL import Image

model.eval()
device = 'cuda' if torch.cuda.is_available() else 'cpu'

train_imgs = sorted((refined / 'images' / 'train').glob('*'))
train_imgs = [f for f in train_imgs if f.suffix.lower() in {'.jpg','.jpeg','.png'}]
print(f'Computing loss for {len(train_imgs)} train images...')

losses = []
BATCH = 8
for i in range(0, len(train_imgs), BATCH):
    batch = train_imgs[i:i+BATCH]
    # ultralytics .predict()로 conf 추출 → 낮은 conf인 GT를 hard로 간주
    results = model.predict(source=[str(b) for b in batch], conf=0.05, iou=0.5, imgsz=1280, verbose=False, save=False)
    for img_path, res in zip(batch, results):
        gt = read_labels(refined / 'labels' / 'train' / (img_path.stem + '.txt'))
        if not gt:
            losses.append((img_path, 0.0))
            continue
        # GT bbox에 대한 모델 conf의 평균 → 낮을수록 hard
        if len(res.boxes) == 0:
            losses.append((img_path, 1.0))  # 검출 0개면 max hard
            continue
        pred_xyxy = res.boxes.xyxy.cpu().numpy()
        pred_conf = res.boxes.conf.cpu().numpy()
        pred_cls = res.boxes.cls.cpu().numpy().astype(int)
        W, H = res.orig_shape[1], res.orig_shape[0]
        confs = []
        for g in gt:
            g_xy = yolo_to_xyxy(g, W, H)
            best_match_conf = 0.0
            for j in range(len(pred_xyxy)):
                if pred_cls[j] == g[0] and iou(g_xy, pred_xyxy[j]) >= 0.3:
                    best_match_conf = max(best_match_conf, pred_conf[j])
            confs.append(best_match_conf)
        avg_conf = sum(confs) / len(confs)
        losses.append((img_path, 1.0 - avg_conf))  # hard score
    if (i+BATCH) % 320 == 0:
        print(f'  {min(i+BATCH, len(train_imgs))}/{len(train_imgs)}')

# 상위 10% hard samples
losses.sort(key=lambda x: x[1], reverse=True)
n_hard = max(1, len(losses) // 10)
hard_samples = losses[:n_hard]
print(f'\nHard samples: {n_hard} ({n_hard/len(losses)*100:.1f}%)')

# Hard samples 5배 복사 (가중치 효과)
WEIGHT = 4  # 4배 추가 = 5배 가중
for img_path, score in hard_samples:
    lbl = refined / 'labels' / 'train' / (img_path.stem + '.txt')
    for k in range(WEIGHT):
        new_name = f'{img_path.stem}_hard{k}{img_path.suffix}'
        shutil.copy2(img_path, refined / 'images' / 'train' / new_name)
        if lbl.exists():
            shutil.copy2(lbl, refined / 'labels' / 'train' / f'{img_path.stem}_hard{k}.txt')

print(f'복사 완료. 새 train 카운트: {len(list((refined / "images" / "train").glob("*")))}')

## 6. 재학습 (multi-stage)

Stage 1: 정제 데이터로 일반 학습 (lr=5e-4, 40ep)
Stage 2: 가장 좋은 best.pt에서 fine-tune (lr=1e-5, 15ep)

In [ ]:
import threading, time

AUTOSAVE = DRIVE_DIR / 'm5v3_autosave'
AUTOSAVE.mkdir(parents=True, exist_ok=True)
PROJECT = Path('/content/runs/m5v3')
PROJECT.mkdir(parents=True, exist_ok=True)

stop_flag = [False]
def autosave_loop():
    while not stop_flag[0]:
        time.sleep(300)
        for s in ['stage1', 'stage2']:
            for src in [PROJECT/s/'weights/last.pt', PROJECT/s/'weights/best.pt']:
                if src.exists():
                    try: shutil.copy2(src, AUTOSAVE / f'{s}_{src.name}')
                    except: pass
        print(f'[autosave] {time.strftime("%H:%M:%S")}')

t = threading.Thread(target=autosave_loop, daemon=True)
t.start()

# Stage 1
print('=== Stage 1: 정제 데이터 학습 (40ep) ===')
model = YOLO('yolo11m.pt')
model.train(
    data=str(data_yaml),
    epochs=40, batch=4, imgsz=1280, cache='disk', workers=4,
    optimizer='AdamW', lr0=5e-4, lrf=0.01, cos_lr=True,
    patience=15, warmup_epochs=3, close_mosaic=15, freeze=0,
    hsv_h=0.015, hsv_s=0.5, hsv_v=0.4,
    degrees=3.0, translate=0.1, scale=0.3,
    mosaic=1.0, mixup=0.1, copy_paste=0.3,
    save_period=5, plots=True,
    project=str(PROJECT), name='stage1', exist_ok=True,
)

stage1_best = PROJECT / 'stage1' / 'weights' / 'best.pt'
print(f'Stage 1 best: {stage1_best}')

# Stage 2: fine-tune
print('\n=== Stage 2: 정제+hard 가중 fine-tune (15ep) ===')
model2 = YOLO(str(stage1_best))
model2.train(
    data=str(data_yaml),
    epochs=15, batch=4, imgsz=1280, cache='disk', workers=4,
    optimizer='AdamW', lr0=1e-5, lrf=0.01, cos_lr=True,
    patience=8, warmup_epochs=1, close_mosaic=10, freeze=10,
    mosaic=0.5, mixup=0.0, copy_paste=0.2,
    save_period=5, plots=True,
    project=str(PROJECT), name='stage2', exist_ok=True,
)
stop_flag[0] = True

In [ ]:
# 두 stage 중 더 좋은 best.pt 선택
stage1_best = PROJECT / 'stage1' / 'weights' / 'best.pt'
stage2_best = PROJECT / 'stage2' / 'weights' / 'best.pt'

best_path = stage2_best if stage2_best.exists() else stage1_best
best_model = YOLO(str(best_path))
best_model.export(format='onnx', opset=17, dynamic=True, simplify=True)
onnx_path = best_path.with_suffix('.onnx')

metrics = best_model.val(data=str(data_yaml), imgsz=1280, batch=4)
print(f'\n=== M5v3 최종 결과 ===')
print(f'mAP50:    {metrics.box.map50:.4f}')
print(f'mAP50-95: {metrics.box.map:.4f}')
print(f'precision: {metrics.box.mp:.4f}')
print(f'recall:    {metrics.box.mr:.4f}')
print(f'0.9? {"YES ✅" if metrics.box.map50 >= 0.9 else "NO"}')

OUT = DRIVE_DIR / 'm5v3_results'
OUT.mkdir(parents=True, exist_ok=True)
shutil.copy2(onnx_path, OUT / 'm5_yolo_seg_frames.onnx')
shutil.copy2(best_path, OUT / 'm5v3_best.pt')
for s in ['stage1', 'stage2']:
    rcsv = PROJECT / s / 'results.csv'
    if rcsv.exists(): shutil.copy2(rcsv, OUT / f'{s}_results.csv')
print('Saved:', OUT)